# 💻 Notebook do Aluno — Aula 08: Interfaces com Gradio e Streamlit + deploy com URL pública

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 08/14 — Módulo 3: Interfaces, Agentes e Integração**  
**⏱️ 1h40min**  
**🌐 Gradio · ngrok · streaming**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Ao final da aula, o RAG do CKP02 está acessível via URL pública — qualquer pessoa com o link consegue fazer perguntas aos documentos do domínio pelo celular ou computador, sem abrir o Colab.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime dos exercícios.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install gradio langchain-community langchain-ollama chromadb pymupdf -q

import gradio as gr, uuid, os
from google.colab import userdata
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# Reutilizar db do CKP02 (já indexado — só recarregar)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db         = Chroma(persist_directory="/content/ckp02", embedding_function=embeddings)

# 👉 LACUNA 1: crie o retriever com k=3
retriever = db.as_retriever(search_kwargs={"k":___})

# 👉 LACUNA 2: monte a chain RAG com memória
store = {}
def obter_hist(sid):
    if sid not in store: store[sid] = ChatMessageHistory()
    return store[sid]

chain_base = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "chat_history": RunnableLambda(lambda _:"")}
    | prompt_com_hist | llm | StrOutputParser()
)
chain_mem = RunnableWithMessageHistory(
    chain_base, obter_hist,
    input_messages_key=___,
    history_messages_key=___,
)

# 👉 LACUNA 3: função de chat com streaming
def chat(msg, hist, sid):
    parcial = ""
    for chunk in chain_mem.stream({"pergunta":___},
                                   config={"configurable":{"session_id":___}}):
        parcial += chunk
        yield parcial

# 👉 LACUNA 4: monte a interface e publique com share=True
with gr.Blocks() as demo:
    sid = gr.State(lambda:str(uuid.uuid4()))
    gr.Markdown("## 📄 DocMind do Grupo — [Nome do Domínio]")
    gr.ChatInterface(fn=___, type="messages", additional_inputs=[___])
demo.launch(share=___)  # True para URL pública

---

## ✍️ Suas anotações

Registre aqui as observações da aula (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 08

Quatro exercícios práticos em sequência — o componente certo do Gradio, streaming com generator, publicação da URL (share/ngrok) e memória multi-sessão — tudo sobre o RAG DocMind do grupo (base do CKP02).

Grupo 3–4 · Google Colab: publique a URL do grupo, poste no canal da turma e teste a URL de outro grupo com 2 perguntas.


### Exercício 1 — Mapa de decisão dos componentes Gradio · ★★☆ · 10 min

*Individual · Colab*

1. Complete o andaime abaixo com o componente do Gradio para o cenário (a) do CKP03 — demo de 1 pergunta → 1 resposta.
2. Complete também os componentes de entrada e de saída de texto.
3. Rode a interface e teste 1 pergunta real do domínio do grupo.
4. Em célula markdown, registre qual componente você escolheria para os cenários (b) chatbot com histórico e (c) app com `gr.State` — e o porquê.

> **💡 Dica:** `gr.Interface` é o componente de 1 entrada → 1 saída; a assinatura `(mensagem, historico)` com `type="messages"` é o que faz o `ChatInterface` detectar um chat.


In [ ]:
# Exercício 1 — o componente certo para o cenário (a) do CKP03
!pip install -q gradio langchain-ollama langchain-community chromadb

import gradio as gr

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a08e1", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough()}
    | ChatPromptTemplate.from_template(
        "<persona>Assistente do domínio. Responda em português.</persona>\n"
        "<contexto>{contexto}</contexto>\n<pergunta>{pergunta}</pergunta>")
    | llm | StrOutputParser()
)

def responder(pergunta: str) -> str:
    return chain_rag.invoke(pergunta)

# 👉 LACUNA 1: componente do Gradio para 1 entrada → 1 saída (sem estado)
demo = gr.___(
    fn=responder,
    # 👉 LACUNA 2: componente de entrada de texto
    inputs=gr.___(label="Pergunta"),
    # 👉 LACUNA 3: componente de saída de texto
    outputs=gr.___(label="Resposta"),
    title="DocMind RAG — Demo do grupo",
)
demo.launch()


### Exercício 2 — Diagnóstico de streaming: return vs. yield · ★★☆ · 10 min

*Individual · Colab*

1. Rode o andaime e observe a versão bloqueante: `invoke` entrega tudo ao final, sem feedback.
2. Complete o método que devolve chunks e o valor enviado a cada passo do generator.
3. Publique a versão com streaming no `gr.ChatInterface` e confirme o texto crescendo token a token.

> **💡 Dica:** o Gradio detecta o generator e ativa o streaming sozinho — não há flag "streaming" para ligar.


In [ ]:
# Exercício 2 — transforme a função em generator
!pip install -q gradio langchain-ollama langchain-community chromadb

import gradio as gr

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a08e2", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough()}
    | ChatPromptTemplate.from_template(
        "<persona>Assistente do domínio. Responda em português.</persona>\n"
        "<contexto>{contexto}</contexto>\n<pergunta>{pergunta}</pergunta>")
    | llm | StrOutputParser()
)

# Evidência no terminal — chunks chegando token a token
for chunk in chain_rag.stream("Qual é o prazo de garantia?"):
    print(chunk, end="|", flush=True)
print()

# Versão bloqueante — devolve tudo ao final (5–30 s sem feedback)
def chat_sem_stream(msg, hist):
    # 👉 LACUNA 1: método que espera a resposta completa
    return chain_rag.___(msg)

# Versão generator — o texto cresce token a token
def chat_com_stream(msg, hist):
    parcial = ""
    # 👉 LACUNA 2: método que devolve chunks
    for chunk in chain_rag.___(msg):
        parcial += chunk
        # 👉 LACUNA 3: envie o estado acumulado do texto
        yield ___

# 👉 LACUNA 4: entregue a função generator à interface
gr.ChatInterface(fn=___, type="messages", title="DocMind RAG — Streaming").launch()


### Exercício 3 — share=True vs. túnel ngrok · ★★☆ · 10 min

*Individual · Colab*

1. Complete a publicação pela opção A (`share=True`) e copie a URL `*.gradio.live` gerada.
2. Depois autentique o túnel ngrok com o token do Colab Secrets e abra a porta 7860.
3. Acesse as duas URLs de outro dispositivo (celular) e compare latência e estabilidade em 2 perguntas.

> **💡 Dica:** `share=True` usa a infraestrutura do Gradio (zero token); o túnel ngrok aponta para a porta do seu app e exige token — nas duas, a URL vive enquanto o runtime do Colab estiver ativo.


In [ ]:
# Exercício 3 — publique pela opção A e depois pelo túnel ngrok
!pip install -q gradio langchain-ollama langchain-community chromadb

import gradio as gr

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a08e3", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough()}
    | ChatPromptTemplate.from_template(
        "<persona>Assistente do domínio. Responda em português.</persona>\n"
        "<contexto>{contexto}</contexto>\n<pergunta>{pergunta}</pergunta>")
    | llm | StrOutputParser()
)

def chat_com_stream(msg, hist):
    parcial = ""
    for chunk in chain_rag.stream(msg):
        parcial += chunk
        yield parcial

chatbot_a = gr.ChatInterface(fn=chat_com_stream, type="messages",
                             title="DocMind RAG — share=True")

# 👉 LACUNA 1: URL pública gerenciada pelo Gradio (zero token)
chatbot_a.launch(share=___)
chatbot_a.close()   # libera o app antes da opção B

from pyngrok import ngrok

# 👉 LACUNA 2: autentique o túnel com o token do Colab Secrets
ngrok.set_auth_token(userdata.get(___))

# 👉 LACUNA 3: abra o túnel na porta 7860
tunel = ngrok.___(7860)
print(f"URL pública do túnel: {tunel.public_url}")

chatbot_b = gr.ChatInterface(fn=chat_com_stream, type="messages",
                             title="DocMind RAG — via ngrok")
chatbot_b.launch(server_port=7860, server_name="0.0.0.0")


### Exercício 4 — Memória multi-sessão publicada (mini-CKP03) · ★★☆ · 10 min

*Individual · Colab*

1. Complete as duas chaves do histórico no `RunnableWithMessageHistory`.
2. Complete o `session_id` no config da função de chat com streaming.
3. Em `gr.Blocks`, complete o `gr.State` e a função entregue ao `ChatInterface`.
4. Publique com `share=True` e valide a memória com perguntas encadeadas tipo "e sobre isso que você disse...".

> **💡 Dica:** o `session_id` é a chave do `store` de históricos — sem o `gr.State`, todas as sessões de browser compartilham o mesmo histórico.


In [ ]:
# Exercício 4 — memória multi-sessão publicada
!pip install -q gradio langchain-ollama langchain-community chromadb

import uuid
import gradio as gr

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a08e4", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

PROMPT_RAG_HIST = """<persona>Assistente do domínio com memória de conversa.</persona>
<historico>{chat_history}</historico>
<contexto>{contexto}</contexto>
<instrucoes>Use o contexto para responder. Se a pergunta se referir ao histórico, use-o.</instrucoes>
<pergunta>{pergunta}</pergunta>"""

chain_com_hist = (
    {"contexto":     retriever | RunnableLambda(formatar_contexto),
     "pergunta":     RunnablePassthrough(),
     "chat_history": RunnableLambda(lambda _: "")}   # preenchido pelo wrapper
    | ChatPromptTemplate.from_template(PROMPT_RAG_HIST)
    | llm | StrOutputParser()
)

store = {}
def obter_hist(sid):
    if sid not in store:
        store[sid] = ChatMessageHistory()
    return store[sid]

# 👉 LACUNA 1: chave do input no prompt do wrapper
# 👉 LACUNA 2: chave do histórico no prompt do wrapper
chain_mem = RunnableWithMessageHistory(
    chain_com_hist, obter_hist,
    input_messages_key=___,
    history_messages_key=___,
)

def chat(msg, hist, sid):
    parcial = ""
    for chunk in chain_mem.stream(
        {"pergunta": msg},
        # 👉 LACUNA 3: a chave da sessão no config
        config={"configurable": {"session_id": ___}},
    ):
        parcial += chunk
        yield parcial

with gr.Blocks(title="DocMind do Grupo") as demo:
    # 👉 LACUNA 4: id único por sessão de browser
    sid = gr.State(lambda: str(___(uuid.uuid4())))
    gr.Markdown("## 📄 DocMind do Grupo — [Nome do Domínio]")
    # 👉 LACUNA 5: entregue a função com memória + o sid como input extra
    gr.ChatInterface(fn=___, type="messages", additional_inputs=[sid])
demo.launch(share=___)   # True → URL pública


## 📚 Referências da aula

- Docs Gradio — ChatInterface, Blocks, streaming e deploy. gradio.app/docs/gradio/chatinterface
- Docs LangChain — RunnableWithMessageHistory para múltiplas sessões. python.langchain.com/docs/how_to/message_history
- Docs ngrok — Túnel HTTP gratuito para desenvolvimento e demos. ngrok.com/docs
- Docs Streamlit — st.chat_message, st.session_state, file_uploader. docs.streamlit.io/develop/api-reference/chat
- Livro Avila, R. D. — Architecting AI Software Systems. Packt, 2025. Cap. 5 — o padrão "Blue and Gold" para deploy seguro de pipelines de IA em produção.
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes inteligentes: a fundamentação teórica da transição de pipeline para agente que acontece nas próximas aulas.

---

**Próxima Aula — Aula 09** — Agentes de IA — ReAct, tools e function calling
  
O chatbot passa a decidir qual ferramenta usar. RAG, web search e calculadora — autonomamente.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*